# SQL query from table names

In This notebook we are going to test if using just the name of the table, and a shord definition of its contect we can use a model like GTP3.5-Turbo to select which tables are necessary to create a SQL Order to answer the user petition.

In [2]:
import os
from google.colab import userdata
from openai import OpenAI
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
  raise ValueError("OPENAI_API_KEY was not found in Colab Secrets.")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

print("OPENAI_API_KEY loaded successfully into the environment.")

OPENAI_API_KEY loaded successfully into the environment.


In [ ]:
#from openai import OpenAI
#import os
#from dotenv import load_dotenv, find_dotenv
#_ = load_dotenv(find_dotenv())

#OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [3]:
#Function to call the model.
def return_OAI(user_message):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)
    context = []
    context.append({'role':'system', "content": user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=context,
            temperature=0,
        )

    return (response.choices[0].message.content)

In [23]:
#Definition of the tables.
import pandas as pd

# Table and definitions sample
data = {'table': ['institution', 'subscription', 'monthly use', 'complaines', 'type of complaines'],
        'definition' : ['name of hospital or clinic, adress, name of each unit',
                        'type of subscription, date of subscription, number of subscription',
                        'number of active subscriptions, number of alerts, number of risky alets',
                        'number of complaines recieved, number of actions dones, number of bad events to solve',
                        'too much false postive alerte, too much false negative alerte, too complexe, ot matching with the unit worklow ']}


df = pd.DataFrame(data)
print(df)
df.head()

                table                                         definition
0         institution  name of hospital or clinic, adress, name of ea...
1        subscription  type of subscription, date of subscription, nu...
2         monthly use  number of active subscriptions, number of aler...
3          complaines  number of complaines recieved, number of actio...
4  type of complaines  too much false postive alerte, too much false ...


,table,definition
0,institution,"name of hospital or clinic, adress, name of ea..."
1,subscription,"type of subscription, date of subscription, nu..."
2,monthly use,"number of active subscriptions, number of aler..."
3,complaines,"number of complaines recieved, number of actio..."
4,type of complaines,"too much false postive alerte, too much false ..."


In [24]:
text_tables = '\n'.join([f"{row['table']}: {row['definition']}" for index, row in df.iterrows()])

In [25]:
print(text_tables)

institution: name of hospital or clinic, adress, name of each unit
subscription: type of subscription, date of subscription, number of subscription
monthly use: number of active subscriptions, number of alerts, number of risky alets
complaines: number of complaines recieved, number of actions dones, number of bad events to solve
type of complaines: too much false postive alerte, too much false negative alerte, too complexe, ot matching with the unit worklow 


In [26]:
prompt_question_tables = """
Given the following tables and their content definitions,
###Tables
{tables}

Tell me which tables would be necessary to query with SQL to address the user's question below.
Return the table names in a json format.
###User Questyion:
{question}
"""


In [27]:
#Creating the prompt, with the user questions and the tables definitions.
pqt1 = prompt_question_tables.format(tables=text_tables, question= "Return the name of the institution with highest number of complaines")

In [28]:
print(return_OAI(pqt1))

```json
{
    "tables": ["institution", "complaines"]
}
```


In [29]:
pqt3 = prompt_question_tables.format(tables=text_tables,
                                     question= "Return the name of the unit with the higher number of alert monthly")

In [30]:
print(return_OAI(pqt3))

```json
{
    "tables": ["monthly use"]
}
```


# Lab Report — SQL Query from Table Names

## What I did
I tested whether GPT-3.5-Turbo could identify the relevant tables needed to answer natural language questions about a fall detection system in an acute geriatric ward. The database includes tables for patients, falls, alerts, staff shifts, and unit activity.

## Versions tested
- **pqt1** — "Return the name of the institution with the highest number of complaints" → correctly returned `["institution", "complaines"]`
- **pqt3** — "Return the name of the unit with the higher number of alert monthly" → returned only `["monthly use"]`

## What worked well
When the question clearly mapped to one or two tables, the model was accurate and fast. It handled simple aggregation questions well.

## Issue observed — incomplete table selection
For pqt3, the model missed the `institution` table, which contains the unit names. It identified the correct data table (`monthly use`) but failed to include the table needed for the JOIN to return a human-readable unit name. **The root cause was the question itself**: the phrasing didn't explicitly signal that a name lookup was required from a separate table.

Rephrasing to *"Return the unit name (from the institution table) with the highest monthly alert count"* likely would have produced the correct two-table result.

## What I learned
- The model is highly literal: it selects tables based on what the question explicitly mentions, not what a SQL JOIN would implicitly require.
- Table definitions matter: a richer description of `institution` (e.g., explicitly stating it holds unit identifiers) could compensate for ambiguous questions.
- Prompt precision is critical in text-to-SQL pipelines — small wording changes have a direct impact on query correctness.

# Exercise
 - Complete the prompts similar to what we did in class.
     - Try a few versions if you have time
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?